In [0]:
%run ../00-common/01.environment-config

In [0]:
from pyspark.sql import functions as F

In [0]:
bronze_table = f'{catalog_name}.{bronze_schema}.circuits'
silver_table = f'{catalog_name}.{silver_schema}.circuits'

**Step 1 - Read bronze `circuits` table**

In [0]:
circuits_df = spark.read.table(bronze_table)

In [0]:
display(circuits_df)

**Step 2 - Keep only the columns required (Drop url column)**

In [0]:
circuits_selected_df = circuits_df.select("circuitId",
                   "circuitName",
                   "lat",
                   "long",
                   "locality",
                   "country",
                   "ingestion_timestamp",
                   "source_file"
                   )

**Step 3 & 4 - Standardise Column Names**

In [0]:
# circuits_renamed_df = (
#    circuits_selected_df
#        .withColumnRenamed("circuitId", "circuit_id")
#        .withColumnRenamed("lat", "latitude")
#        .withColumnRenamed("long", "longitude")
#        .withColumnRenamed("circuitName", "circuit_name")
#)

In [0]:
circuits_renamed_df = (
    circuits_selected_df
        .withColumnsRenamed({"circuitId":"circuit_id",
                            "lat":"latitude",
                            "long":"longitude",
                            "circuitName":"circuit_name"
        })
)

**Step 5 - Filter out rows where circuit_id is null (business key validation)**

In [0]:
circuits_valid_df = circuits_renamed_df.filter(
    F.col("circuit_id").isNotNull()
)

**Step 6 - Remove duplicate records**

In [0]:
circuits_distinct_df = circuits_valid_df.dropDuplicates(["circuit_id"])

In [0]:
display(circuits_distinct_df)

**Step 7 - Transform values of columns `circuit_name` and `locality` to Title Case**

In [0]:
circuits_final_df = (
    circuits_distinct_df
        .withColumn("circuit_name", F.initcap(F.col("circuit_name")))
        .withColumn("locality", F.initcap(F.col("locality")))
)

**Step 8 - Write the transformed data to silver `circuits` table**

In [0]:
(
    circuits_final_df
        .write
        .format('delta')
        .mode('overwrite')
        .saveAsTable(silver_table)
)

In [0]:
%sql
select * from formula1.silver.circuits